In [1]:
import torch
import torch.nn as nn
from torchvision import models, transforms, datasets
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# ----- Paramètres -----
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMAGE_SIZE = 224  # compatible VGG16 et EfficientNet
NUM_CLASSES = 4   # adapte selon ton dataset
TEST_DATA_PATH = "chemin/vers/test"  # dossier contenant les images de test
VGG16_PATH = "chemin/vers/vgg16.pth"
EFFNET_PATH = "chemin/vers/efficientnet.pth"

# ----- Transformations pour les images -----
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

# ----- Dataset et DataLoader de test -----
test_dataset = datasets.ImageFolder(root=TEST_DATA_PATH, transform=transform)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ----- Charger les modèles -----
# VGG16
vgg16_model = models.vgg16(pretrained=False)
vgg16_model.classifier[6] = nn.Linear(4096, NUM_CLASSES)
vgg16_model.load_state_dict(torch.load(VGG16_PATH, map_location=DEVICE))
vgg16_model.to(DEVICE)
vgg16_model.eval()

# EfficientNet
efficient_model = models.efficientnet_b0(pretrained=False)
efficient_model.classifier[1] = nn.Linear(efficient_model.classifier[1].in_features, NUM_CLASSES)
efficient_model.load_state_dict(torch.load(EFFNET_PATH, map_location=DEVICE))
efficient_model.to(DEVICE)
efficient_model.eval()

# ----- Fonction d'évaluation -----
def evaluate_model(model, loader):
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    metrics = {
        "accuracy": accuracy_score(all_labels, all_preds),
        "precision": precision_score(all_labels, all_preds, average='macro'),
        "recall": recall_score(all_labels, all_preds, average='macro'),
        "f1_score": f1_score(all_labels, all_preds, average='macro')
    }
    return metrics

# ----- Évaluer les modèles -----
results = {
    "model": ["VGG16", "EfficientNet"],
    "accuracy": [],
    "precision": [],
    "recall": [],
    "f1_score": []
}

for name, model in zip(["VGG16", "EfficientNet"], [vgg16_model, efficient_model]):
    metrics = evaluate_model(model, test_loader)
    results["accuracy"].append(metrics["accuracy"])
    results["precision"].append(metrics["precision"])
    results["recall"].append(metrics["recall"])
    results["f1_score"].append(metrics["f1_score"])

# ----- Sauvegarder les résultats -----
df = pd.DataFrame(results)
df.to_csv("evaluation_results.csv", index=False)
print("Évaluation terminée. Résultats enregistrés dans evaluation_results.csv")


FileNotFoundError: [WinError 3] Le chemin d’accès spécifié est introuvable: 'chemin/vers/test'